# InGaN QW / PL 시뮬레이션

세 코드 셀을 순서대로 실행하세요. 두 번째 셀의 **IN_FRACTION** 값만 바꾸면 QW의 In 비율을 조정할 수 있습니다.
`0.20`은 In 20%, `0.25`는 In 25%입니다. 반복 계산은 두 번째·세 번째 셀을 실행하세요.
QW·PL 그래프를 새 결과에서 표시하며, PL 최대 세기는 **1.0 a.u.**로 정규화됩니다.


In [ ]:
from pathlib import Path
import subprocess
import sys

repo = Path("/content/PL_simulation")
repo_url = "https://github.com/Say1Won/PL_simulation.git"

if not (repo / ".git").is_dir():
    subprocess.run(["git", "clone", "--branch", "main", repo_url, str(repo)], check=True)
else:
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only", "origin", "main"], check=True)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(repo / "requirements.txt")],
    check=True,
)
commit = subprocess.check_output(["git", "-C", str(repo), "rev-parse", "--short", "HEAD"], text=True).strip()
print(f"사용할 코드 버전: {commit}")


In [ ]:
IN_FRACTION = 0.20  # In 20%; 25%로 계산하려면 0.25

output_root = repo / "outputs"
previous_runs = set(output_root.glob("run_*"))
subprocess.run(
    [sys.executable, str(repo / "main.py"),
     "--indium-fraction", str(IN_FRACTION), "--output-root", str(output_root)],
    cwd=repo,
    check=True,
)
new_runs = set(output_root.glob("run_*")) - previous_runs
if len(new_runs) != 1:
    raise RuntimeError("이번 실행의 결과 폴더를 확인할 수 없습니다.")
run_dir = new_runs.pop()
print(f"이번 실행 결과: {run_dir}")


In [ ]:
import json
from IPython.display import Image, display

summary = json.loads((run_dir / "analysis/pl_summary.json").read_text(encoding="utf-8"))
if summary["intensity_unit"] != "a.u." or summary["intensity"] != 1.0:
    raise RuntimeError("PL 최대 세기가 1.0 a.u.가 아닙니다. 첫 셀에서 최신 코드를 받아 다시 실행하세요.")

applied_fraction = summary["calculation_metadata"]["structure"]["layers"][1]["indium_fraction"]
print(f"계산에 사용한 In 비율: {applied_fraction:g} ({100 * applied_fraction:g}%)")
print("PL 최대 세기: 1.0 a.u.")
display(Image(filename=str(run_dir / "figures/quantum_well.png")))
display(Image(filename=str(run_dir / "figures/pl_spectrum.png")))
print(f"CSV: {run_dir / 'analysis/pl_spectrum.csv'}")
